In [11]:
import csv
from faker import Faker
import random

# Khởi tạo Faker
fake = Faker()

def generate_customer_feedback(custom_products=None, feedbacks_pool=None):
    # Chọn ngẫu nhiên một feedback từ danh sách pool
    feedback = random.choice(feedbacks_pool)
    
    # Tạo tên sản phẩm ngẫu nhiên nếu không cung cấp custom_products
    product = random.choice(custom_products) if custom_products else fake.word()

    return {
        "customer_name": fake.name(),
        "product_name": product,
        "rating": random.randint(1, 5),
        "feedback": feedback,
        "purchase_date": fake.date_between(start_date="-1y", end_date="today"),
        "location": fake.city()
    }

# Danh sách feedbacks ban đầu
feedback_comments = [
    "Product quality is good.",
    "Delivery took too long, I had to wait a week to receive it. What kind of service is this?",
    "Fast delivery, the product matches the description.",
    "The product is ugly, poor quality, you get what you pay for. I'm never buying from this shop again!",
    "The price is a bit high, but the quality is decent.",
    "The product did not meet my expectations, improvements are needed.",
    "So disappointed. Ordered a premium product, but what I got is complete crap. This shop needs to review their business practices!",
    "Customer service is excellent, I will definitely buy again.",
    "Delivery was slow, but the product is acceptable.",
    "The product is defective, I have requested a return.",
    "If you are going to deliver like this, just shut down your business! This is the worst shop  have ever dealt with.",
    "The product is very good, and delivery was faster than expected. I will continue to support this shop!",
    "Damn it, I spent 5 million and received such garbage? What kind of stupid business is this? Never coming back!",
    "This is my third purchase, and the quality is always consistent. Definitely worth the money!",
    "Is this a scam? The ad looked amazing, but the product I received is absolute trash. Stop selling stuff like this!",
    "The shop is very thoughtful, packaging is careful. The product matches the picture, nothing to complain about.",
    "Amazing! The product is exactly as described, and the price is reasonable. Thank you, shop!"
]

# Tạo danh sách sản phẩm ngẫu nhiên
num_products = 50  # Số lượng sản phẩm cần tạo
random_products = [
    fake.word().capitalize() + " " + random.choice(["Pro", "Max", "Lite", "Edition"])
    for _ in range(num_products)
]

# Số lượng phản hồi cần tạo
num_feedbacks = 1000  # Số feedback cần tạo
train_ratio = 0.8  # Tỷ lệ dữ liệu train

# Tăng cường feedbacks khi không đủ
if len(feedback_comments) < num_feedbacks:
    feedbacks_pool = feedback_comments.copy()
    while len(feedbacks_pool) < num_feedbacks:
        feedbacks_pool.extend(feedback_comments)
    feedbacks_pool = feedbacks_pool[:num_feedbacks]
else:
    feedbacks_pool = feedback_comments.copy()

# Sinh dữ liệu phản hồi khách hàng
customer_feedbacks = [
    generate_customer_feedback(custom_products=random_products, feedbacks_pool=feedbacks_pool)
    for _ in range(num_feedbacks)
]

In [12]:
# Chia dữ liệu train/test
num_train = int(num_feedbacks * train_ratio)
num_test = num_feedbacks - num_train

train_feedbacks = customer_feedbacks[:num_train]
test_feedbacks = customer_feedbacks[num_train:]

# Xuất dữ liệu ra file CSV
train_csv_path = "customer_feedbacks_train.csv"
test_csv_path = "customer_feedbacks_test.csv"



In [13]:
# Ghi file train
with open(train_csv_path, mode="w", encoding="utf-8", newline="") as train_file:
    writer = csv.DictWriter(train_file, fieldnames=["customer_name", "product_name", "rating", "feedback", "purchase_date", "location"])
    writer.writeheader()
    writer.writerows(train_feedbacks)

# Ghi file test
with open(test_csv_path, mode="w", encoding="utf-8", newline="") as test_file:
    writer = csv.DictWriter(test_file, fieldnames=["customer_name", "product_name", "rating", "feedback", "purchase_date", "location"])
    writer.writeheader()
    writer.writerows(test_feedbacks)


In [14]:
import spacy
import csv

# Load spaCy model để xử lý NER
nlp = spacy.load("en_core_web_sm")  

# Chạy file csvcsv
train_csv_path = "customer_feedbacks_train.csv"
train_ner_output_path = "customer_feedbacks_train_ner.csv"

test_csv_path = "customer_feedbacks_test.csv"
test_ner_output_path = "customer_feedbacks_test_ner.csv"

# Đọc file train
with open(train_csv_path, mode="r", encoding="utf-8") as train_file:
    reader = csv.DictReader(train_file)  
    train_data = list(reader)  



In [ ]:
# Tạo danh sách chỉ chứa cột 'ner_output' cho dữ liệu train
train_ner_data = []

# Thêm cột mới để lưu kết quả NER cho dữ liệu train
for row in train_data:
    # Tạo đoạn văn bản đầu vào từ các cột dữ liệu
    input_text = f"{row['customer_name']} bought {row['product_name']} and said: {row['feedback']} in {row['location']}."
    
    # Áp dụng NER
    doc = nlp(input_text)
    
    # Tạo đoạn văn bản mới với các thực thể được đánh dấu
    formatted_output = input_text  # Khởi tạo đoạn văn bản từ input_text
    for ent in doc.ents:
        formatted_output = formatted_output.replace(ent.text, f"<{ent.label_.lower()}> {ent.text}")  # Thay thế thực thể
    
    # Lưu kết quả vào danh sách mới với chỉ cột 'ner_output'
    train_ner_data.append({"ner_output": formatted_output})


In [ ]:
# Lưu dữ liệu mới vào file train output với chỉ cột 'ner_output'
with open(train_ner_output_path, mode="w", encoding="utf-8", newline="") as train_output_file:
    writer = csv.DictWriter(train_output_file, fieldnames=["ner_output"])  # Chỉ ghi cột 'ner_output'
    writer.writeheader()  # Ghi header
    writer.writerows(train_ner_data)  # Ghi tất cả các dòng dữ liệu mới

In [15]:
# Đọc file test
with open(test_csv_path, mode="r", encoding="utf-8") as test_file:
    reader = csv.DictReader(test_file)
    test_data = list(reader)

# Tạo danh sách chỉ chứa cột 'ner_output' cho dữ liệu test
test_ner_data = []

# Thêm cột mới để lưu kết quả NER cho dữ liệu test
for row in test_data:
    input_text = f"{row['customer_name']} bought {row['product_name']} and said: {row['feedback']} in {row['location']}."
    doc = nlp(input_text)
    formatted_output = input_text
    for ent in doc.ents:
        formatted_output = formatted_output.replace(ent.text, f"<{ent.label_.lower()}> {ent.text}")
    test_ner_data.append({"ner_output": formatted_output})

# Lưu dữ liệu mới vào file test output với chỉ cột 'ner_output'
with open(test_ner_output_path, mode="w", encoding="utf-8", newline="") as test_output_file:
    writer = csv.DictWriter(test_output_file, fieldnames=["ner_output"])  # Chỉ ghi cột 'ner_output'
    writer.writeheader()
    writer.writerows(test_ner_data)


In [1]:
import pandas as pd
import spacy
from sklearn.model_selection import train_test_split
from datasets import Dataset

# Đọc dữ liệu từ file CSV
train_df = pd.read_csv("customer_feedbacks_train_ner.csv")
test_df = pd.read_csv("customer_feedbacks_test_ner.csv")

# Tiền xử lý dữ liệu NER, tách entity và label
def process_data(df):
    texts = []
    labels = []
    
    for _, row in df.iterrows():
        text = row['ner_output']
        entities = []
        current_entity = []
        current_label = None
        for word in text.split():
            if word.startswith("<") and word.endswith(">"):
                if current_entity:
                    entities.append((" ".join(current_entity), current_label))
                    current_entity = []
                current_label = word.strip("<>").lower()
            else:
                current_entity.append(word)
        
        # entity cuối cùng 
        if current_entity:
            entities.append((" ".join(current_entity), current_label))

        texts.append(" ".join([word for word, _ in entities]))
        labels.append([label for _, label in entities])

    return texts, labels

# Xử lý train và test data
train_texts, train_labels = process_data(train_df)
test_texts, test_labels = process_data(test_df)

# Chuyển dữ liệu thành dataset
train_dataset = Dataset.from_dict({"text": train_texts, "labels": train_labels})
test_dataset = Dataset.from_dict({"text": test_texts, "labels": test_labels})



c:\Users\trant\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
import pandas as pd
from datasets import Dataset
from transformers import BertTokenizerFast, BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Đọc dữ liệu từ file CSV
df = pd.read_csv("customer_feedbacks_train.csv", header=None)

# Chỉ giữ lại cột 'text' và 'rating'
df = df[["text", "rating"]]

df = df.sample(frac=0.1, random_state=42).reset_index(drop=True)

# Mã hóa nhãn (rating) thành số nguyên
label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["rating"])  

# Chuyển đổi thành Dataset của thư viện `datasets`
dataset = Dataset.from_pandas(df)

# Tokenizer và mô hình
tokenizer = BertTokenizerFast.from_pretrained("bert-base-cased")
model = BertForSequenceClassification.from_pretrained("bert-base-cased", num_labels=len(df["label"].unique()))

# Tokenization
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

dataset = dataset.map(tokenize_function, batched=True)
dataset = dataset.remove_columns(["text", "rating"])
dataset.set_format("torch")

# Chia dữ liệu thành train và eval
train_test_split = dataset.train_test_split(test_size=0.2)
train_dataset = train_test_split["train"]
eval_dataset = train_test_split["test"]

# Định nghĩa hàm đánh giá
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = predictions.argmax(axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

# Thiết lập tham số huấn luyện
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
)

# Huấn luyện mô hình
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

# Đánh giá mô hình
results = trainer.evaluate()
print(results)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.

Map: 100%|██████████| 80/80 [00:00<00:00, 1361.68 examples/s]
c:\Users\trant\AppData\Local\Programs\Python\Python310\lib\site-packages\transformers\training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
  0%|          | 0/5 [02:32<?, ?it/s]


c:\Users\trant\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(resul

{'eval_loss': 1.7001218795776367, 'eval_accuracy': 0.125, 'eval_precision': 0.16666666666666666, 'eval_recall': 0.125, 'eval_f1': 0.14285714285714285, 'eval_runtime': 2.8726, 'eval_samples_per_second': 5.57, 'eval_steps_per_second': 0.696, 'epoch': 1.0}




c:\Users\trant\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


                                               
                                            
  1%|▏         | 3/240 [03:42<33:09,  8.39s/it]


{'eval_loss': 1.7687230110168457, 'eval_accuracy': 0.0, 'eval_precision': 0.0, 'eval_recall': 0.0, 'eval_f1': 0.0, 'eval_runtime': 3.145, 'eval_samples_per_second': 5.087, 'eval_steps_per_second': 0.636, 'epoch': 2.0}




c:\Users\trant\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


                                               
                                            
  1%|▏         | 3/240 [04:31<33:09,  8.39s/it]
                                               
100%|██████████| 24/24 [02:48<00:00,  7.01s/it]


{'eval_loss': 1.7764461040496826, 'eval_accuracy': 0.0, 'eval_precision': 0.0, 'eval_recall': 0.0, 'eval_f1': 0.0, 'eval_runtime': 2.3779, 'eval_samples_per_second': 6.729, 'eval_steps_per_second': 0.841, 'epoch': 3.0}
{'train_runtime': 168.5498, 'train_samples_per_second': 1.139, 'train_steps_per_second': 0.142, 'train_loss': 1.5506590207417805, 'epoch': 3.0}


c:\Users\trant\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
100%|██████████| 2/2 [00:01<00:00,  1.72it/s]

{'eval_loss': 1.7764461040496826, 'eval_accuracy': 0.0, 'eval_precision': 0.0, 'eval_recall': 0.0, 'eval_f1': 0.0, 'eval_runtime': 2.1799, 'eval_samples_per_second': 7.34, 'eval_steps_per_second': 0.917, 'epoch': 3.0}


In [7]:
# Định nghĩa đánh giá Precision, Recall, F1
def compute_metrics(p):
    predictions, labels = p
    predictions = predictions.argmax(axis=-1)  
    true_labels = labels.flatten()
    pred_labels = predictions.flatten()

    # Bỏ qua token bị gán giá trị -100 (padding token)
    valid_indices = true_labels != -100
    true_labels = true_labels[valid_indices]
    pred_labels = pred_labels[valid_indices]

    precision, recall, f1, _ = precision_recall_fscore_support(true_labels, pred_labels, average="micro")
    return {"precision": precision, "recall": recall, "f1": f1}

# Định nghĩa các tham số huấn luyện
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    save_total_limit=2,  # Giới hạn số lượng checkpoint được lưu
    logging_steps=100   # Tần suất logging
)

# Huấn luyện mô hình với Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

# Huấn luyện mô hình
trainer.train()

# Đánh giá mô hình trên tập test
metrics = trainer.evaluate()
print(metrics)

c:\Users\trant\AppData\Local\Programs\Python\Python310\lib\site-packages\transformers\training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(





                                               
                                           
  1%|▏         | 3/240 [11:04<33:09,  8.39s/it]


{'eval_runtime': 2.8281, 'eval_samples_per_second': 7.072, 'eval_steps_per_second': 1.061, 'epoch': 1.0}







                                               
                                            
  1%|▏         | 3/240 [11:51<33:09,  8.39s/it]


{'eval_runtime': 2.402, 'eval_samples_per_second': 8.326, 'eval_steps_per_second': 1.249, 'epoch': 2.0}







                                               
                                            
  1%|▏         | 3/240 [12:34<33:09,  8.39s/it]
                                               
100%|██████████| 24/24 [02:19<00:00,  5.83s/it]


{'eval_runtime': 2.7963, 'eval_samples_per_second': 7.152, 'eval_steps_per_second': 1.073, 'epoch': 3.0}
{'train_runtime': 139.8659, 'train_samples_per_second': 1.373, 'train_steps_per_second': 0.172, 'train_loss': 1.4295439720153809, 'epoch': 3.0}


100%|██████████| 3/3 [00:01<00:00,  1.63it/s]

{'eval_runtime': 2.6954, 'eval_samples_per_second': 7.42, 'eval_steps_per_second': 1.113, 'epoch': 3.0}


In [8]:
model.save_pretrained("./ner_model")
tokenizer.save_pretrained("./ner_model")


('./ner_model\\tokenizer_config.json',
 './ner_model\\special_tokens_map.json',
 './ner_model\\vocab.txt',
 './ner_model\\added_tokens.json',
 './ner_model\\tokenizer.json')